# Lizard Image Classification Pipeline - Kaggle Final Training

Deze Kaggle-versie is bewust compact gehouden, maar bevat wel de Optuna-stap die nodig is om de finale trainingsparameters te kiezen. De onderzoeks-, visualisatie-, cleaning- en explainability-stukken uit de gewone pipeline zijn hier weggelaten zodat Kaggle vooral bezig is met tunen, final trainen en `submission.csv` maken.

## 1. Setup

In [ ]:
# Kaggle heeft de meeste packages normaal al. Zet deze aan als Optuna ontbreekt.
# %pip install -q optuna

In [ ]:
from pathlib import Path
import gc
import random
import warnings

import numpy as np
import optuna
import pandas as pd
from PIL import ImageFile
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

try:
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision enabled.")
except Exception as exc:
    print("GPU setup skipped:", exc)

In [ ]:
def show_kaggle_input_tree(max_lines=80):
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        print("/kaggle/input does not exist in this runtime.")
        return
    shown = 0
    for path in sorted(kaggle_input.rglob("*")):
        depth = len(path.relative_to(kaggle_input).parts)
        if depth > 4:
            continue
        print("  " * depth + path.name + ("/" if path.is_dir() else ""))
        shown += 1
        if shown >= max_lines:
            print("...")
            break


def candidate_data_roots():
    candidates = [
        Path("/kaggle/input/lizard-better/AI-Lizard-Prediction-Challenge"),
        Path("/kaggle/input/lizard-better"),
        Path.cwd(),
        Path.cwd() / "AI-Lizard-Prediction-Challenge",
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(path for path in kaggle_input.rglob("*") if path.is_dir())
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        if resolved not in seen:
            seen.add(resolved)
            yield resolved


def find_data_root():
    for candidate in candidate_data_roots():
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    show_kaggle_input_tree()
    raise FileNotFoundError(
        "Could not find a data root containing train/ and test/. Attach the lizard dataset to this Kaggle notebook."
    )


def find_input_file(filename, data_root):
    search_roots = [data_root, data_root.parent, Path("/kaggle/input"), Path.cwd()]
    for root in search_roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct.resolve()
        if root.name == "input":
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    return data_root / filename


DATA_ROOT = find_data_root()
WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_ROOT

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
TRAIN_CSV = find_input_file("train.csv", DATA_ROOT)
TEST_CSV = find_input_file("test.csv", DATA_ROOT)

IMG_SIZE = (260, 260)
BACKBONE_NAME = "convnext_tiny"
VAL_SIZE = 0.15
BATCH_SIZE = 24
AUTOTUNE = tf.data.AUTOTUNE

HAS_GPU = len(tf.config.list_physical_devices("GPU")) > 0
RUN_OPTUNA = True
N_OPTUNA_TRIALS = 20 if not HAS_GPU else 30
N_FOLDS = 3 if not HAS_GPU else 5
OPTUNA_HEAD_MAX_EPOCHS = 8 if not HAS_GPU else 12
OPTUNA_SEARCH_FINE_TUNING = HAS_GPU
OPTUNA_FINE_TUNE_MAX_EPOCHS = 0 if not HAS_GPU else 8
FINAL_HEAD_EPOCHS = 8
FINAL_FORCE_FINE_TUNING = True
FINAL_FINE_TUNE_LAST_N_LAYERS = 40
FINAL_FINE_TUNE_EPOCHS = 12
FINAL_FINE_TUNE_LEARNING_RATE = 5e-5
USE_TTA = True

OPTUNA_BATCH_CHOICES = [16, 24, 32]
OPTUNA_DENSE_CHOICES = [0, 64, 128, 256]

print("Data root:", DATA_ROOT)
print("Working root:", WORKING_ROOT)
print("Run Optuna:", RUN_OPTUNA)
print("Optuna trials:", N_OPTUNA_TRIALS)
print("Optuna folds:", N_FOLDS)
print("Optuna fine-tuning search:", OPTUNA_SEARCH_FINE_TUNING)

assert TRAIN_DIR.exists(), f"Missing training directory: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"Missing test directory: {TEST_DIR}"
assert TRAIN_CSV.exists(), f"Missing train CSV: {TRAIN_CSV}"
assert TEST_CSV.exists(), f"Missing test CSV: {TEST_CSV}"

## 2. Data Laden

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def list_image_files(directory, extensions=IMAGE_EXTENSIONS):
    return sorted(path for path in Path(directory).rglob("*") if path.suffix.lower() in extensions)


def build_image_lookup(directory):
    return {path.name.casefold(): path for path in list_image_files(directory)}


def resolve_image_path(image_id, lookup):
    image_id = str(image_id)
    candidates = [image_id, Path(image_id).name]
    if Path(image_id).suffix == "":
        candidates.extend(f"{image_id}{extension}" for extension in IMAGE_EXTENSIONS)
    for candidate in candidates:
        resolved = lookup.get(candidate.casefold())
        if resolved is not None:
            return resolved
    return None


train_csv = pd.read_csv(TRAIN_CSV)
test_csv = pd.read_csv(TEST_CSV)
train_lookup = build_image_lookup(TRAIN_DIR)
test_lookup = build_image_lookup(TEST_DIR)

train_csv["filepath"] = train_csv["id"].map(lambda image_id: resolve_image_path(image_id, train_lookup))
missing_train = train_csv[train_csv["filepath"].isna()]
if len(missing_train) > 0:
    print(f"Skipping {len(missing_train)} train rows without a matching image file.")
df = train_csv[train_csv["filepath"].notna()].copy().reset_index(drop=True)
df["filepath"] = df["filepath"].astype(str)
df["label"] = df["label"].astype(int)

class_labels = sorted(df["label"].unique())
class_names = [
    df.loc[df["label"] == label, "filepath"].map(lambda path: Path(path).parent.name).mode().iloc[0]
    for label in class_labels
]
id_to_class = {label: class_name for label, class_name in zip(class_labels, class_names)}

test_df = test_csv.copy()
test_df["filepath"] = test_df["id"].map(lambda image_id: resolve_image_path(image_id, test_lookup))
assert test_df["filepath"].notna().all(), "Some test ids could not be matched to image files."
test_df["filepath"] = test_df["filepath"].astype(str)

train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df["label"],
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training rows: {len(df):,}")
print(f"Train split: {len(train_df):,}")
print(f"Validation split: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}")
print("Class mapping:", id_to_class)

## 3. TensorFlow Datasets

In [ ]:
def decode_and_resize(path, label=None, img_size=IMG_SIZE):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, img_size, method=tf.image.ResizeMethod.BICUBIC)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(image, 0.0, 255.0)
    if label is None:
        return image
    return image, tf.cast(label, tf.int32)


def make_dataset(dataframe, batch_size=BATCH_SIZE, shuffle=False):
    paths = dataframe["filepath"].astype(str).values
    labels = dataframe["label"].astype(np.int32).values
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    return dataset.map(decode_and_resize, num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)


def make_inference_dataset(dataframe, batch_size=BATCH_SIZE):
    paths = dataframe["filepath"].astype(str).values
    dataset = tf.data.Dataset.from_tensor_slices(paths)
    return dataset.map(lambda path: decode_and_resize(path), num_parallel_calls=AUTOTUNE).batch(batch_size).prefetch(AUTOTUNE)


def make_class_weight_dict(labels):
    classes = np.array(sorted(pd.Series(labels).unique()))
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=np.asarray(labels))
    return {int(class_id): float(weight) for class_id, weight in zip(classes, weights)}

## 4. Model Helpers

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
    ],
    name="augmentation",
)

BACKBONE_LAYER_NAMES = {
    "convnext_tiny": "convnext_tiny",
    "efficientnetv2b0": "efficientnetv2-b0",
    "efficientnetv2s": "efficientnetv2-s",
}
BACKBONE_BUILDERS = {
    "convnext_tiny": keras.applications.ConvNeXtTiny,
    "efficientnetv2b0": keras.applications.EfficientNetV2B0,
    "efficientnetv2s": keras.applications.EfficientNetV2S,
}
BACKBONE_LAYER_NAME = BACKBONE_LAYER_NAMES[BACKBONE_NAME]


def get_optimizer(name="adam", learning_rate=3e-4, weight_decay=1e-4):
    name = name.lower()
    if name == "adamw":
        try:
            return keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay)
        except AttributeError:
            print("AdamW unavailable; falling back to Adam.")
            return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    raise ValueError(f"Unsupported optimizer: {name}")


def make_backbone(backbone_name, img_size):
    builder = BACKBONE_BUILDERS[backbone_name]
    try:
        return builder(
            include_top=False,
            include_preprocessing=True,
            weights="imagenet",
            input_shape=(*img_size, 3),
            pooling="avg",
            name=BACKBONE_LAYER_NAME,
        )
    except Exception as exc:
        print(f"Could not load ImageNet weights: {exc}")
        print("Falling back to random initialization.")
        return builder(
            include_top=False,
            include_preprocessing=True,
            weights=None,
            input_shape=(*img_size, 3),
            pooling="avg",
            name=BACKBONE_LAYER_NAME,
        )


def compile_classifier_model(model, learning_rate, optimizer_name="adam", weight_decay=1e-4):
    model.compile(
        optimizer=get_optimizer(optimizer_name, learning_rate, weight_decay),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )


def set_backbone_fine_tuning(model, fine_tune_last_n_layers=0):
    backbone = model.get_layer(BACKBONE_LAYER_NAME)
    if fine_tune_last_n_layers <= 0:
        backbone.trainable = False
        for layer in backbone.layers:
            layer.trainable = False
        return

    backbone.trainable = True
    for layer in backbone.layers[:-fine_tune_last_n_layers]:
        layer.trainable = False
    for layer in backbone.layers[-fine_tune_last_n_layers:]:
        layer.trainable = True
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False


def build_model(
    num_classes,
    dense_units=0,
    dropout_rate=0.35,
    learning_rate=3e-4,
    optimizer_name="adam",
    weight_decay=1e-4,
):
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
    x = data_augmentation(inputs)
    backbone = make_backbone(BACKBONE_NAME, IMG_SIZE)
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.BatchNormalization()(x)
    if dense_units > 0:
        x = layers.Dense(dense_units, activation="relu")(x)
        x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="class_probabilities")(x)
    model = keras.Model(inputs, outputs, name=f"lizard_{BACKBONE_NAME}")
    compile_classifier_model(model, learning_rate, optimizer_name, weight_decay)
    return model

## 5. Optuna Hyperparameter Tuning

In [ ]:
class ValidationMacroF1Callback(keras.callbacks.Callback):
    def __init__(self, validation_data, validation_labels):
        super().__init__()
        self.validation_data = validation_data
        self.validation_labels = np.asarray(validation_labels)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        probabilities = self.model.predict(self.validation_data, verbose=0)
        predictions = probabilities.argmax(axis=1)
        macro_f1 = float(f1_score(self.validation_labels, predictions, average="macro"))
        logs["val_macro_f1"] = macro_f1


def make_cosine_decay_callback(initial_lr, epochs, min_lr=1e-6, verbose=0):
    initial_lr = float(initial_lr)
    min_lr = float(min_lr)
    total_decay_steps = max(int(epochs) - 1, 1)

    def cosine_decay(epoch, current_lr):
        progress = min(epoch, total_decay_steps) / total_decay_steps
        cosine = 0.5 * (1.0 + np.cos(np.pi * progress))
        return float(min_lr + (initial_lr - min_lr) * cosine)

    return keras.callbacks.LearningRateScheduler(cosine_decay, verbose=verbose)


def make_tuning_callbacks(validation_data, validation_labels, initial_lr, epochs, patience=3):
    return [
        ValidationMacroF1Callback(validation_data, validation_labels),
        make_cosine_decay_callback(initial_lr, epochs, verbose=0),
        keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=0,
        ),
    ]


def objective(trial):
    trial_params = {
        "learning_rate": trial.suggest_float("learning_rate", 2e-4, 8e-4, log=True),
        "dropout_rate": trial.suggest_float("dropout_rate", 0.35, 0.60),
        "dense_units": trial.suggest_categorical("dense_units", OPTUNA_DENSE_CHOICES),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw"]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", OPTUNA_BATCH_CHOICES),
        "use_fine_tuning": trial.suggest_categorical(
            "use_fine_tuning", [False, True] if OPTUNA_SEARCH_FINE_TUNING else [False]
        ),
        "head_epochs": OPTUNA_HEAD_MAX_EPOCHS,
    }

    if trial_params["use_fine_tuning"]:
        trial_params.update({
            "fine_tune_last_n_layers": trial.suggest_categorical("fine_tune_last_n_layers", [20, 40, 80]),
            "fine_tune_learning_rate": trial.suggest_float("fine_tune_learning_rate", 5e-6, 8e-5, log=True),
            "fine_tune_epochs": trial.suggest_int("fine_tune_epochs", 3, OPTUNA_FINE_TUNE_MAX_EPOCHS),
        })
    else:
        trial_params.update({
            "fine_tune_last_n_layers": 0,
            "fine_tune_learning_rate": 1e-6,
            "fine_tune_epochs": 0,
        })

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []

    print(f"Trial {trial.number}: lr={trial_params['learning_rate']:.2e}, "
          f"dropout={trial_params['dropout_rate']:.2f}, "
          f"dense={trial_params['dense_units']}, "
          f"optimizer={trial_params['optimizer']}", flush=True)

    for fold, (fold_train_idx, fold_val_idx) in enumerate(skf.split(train_df, train_df["label"])):
        print(f"Trial {trial.number} - fold {fold + 1}/{N_FOLDS}", flush=True)

        fold_train_df = train_df.iloc[fold_train_idx].reset_index(drop=True)
        fold_val_df = train_df.iloc[fold_val_idx].reset_index(drop=True)
        fold_train_ds = make_dataset(fold_train_df, batch_size=trial_params["batch_size"], shuffle=True)
        fold_val_ds = make_dataset(fold_val_df, batch_size=trial_params["batch_size"])
        fold_val_labels = fold_val_df["label"].to_numpy()
        fold_class_weights = make_class_weight_dict(fold_train_df["label"].values)

        model = build_model(
            num_classes=len(class_names),
            dense_units=int(trial_params["dense_units"]),
            dropout_rate=float(trial_params["dropout_rate"]),
            learning_rate=float(trial_params["learning_rate"]),
            optimizer_name=trial_params["optimizer"],
            weight_decay=float(trial_params["weight_decay"]),
        )

        model.fit(
            fold_train_ds,
            validation_data=fold_val_ds,
            epochs=int(trial_params["head_epochs"]),
            callbacks=make_tuning_callbacks(
                fold_val_ds,
                fold_val_labels,
                initial_lr=float(trial_params["learning_rate"]),
                epochs=int(trial_params["head_epochs"]),
            ),
            class_weight=fold_class_weights,
            verbose=0,
        )

        if trial_params["use_fine_tuning"]:
            set_backbone_fine_tuning(model, int(trial_params["fine_tune_last_n_layers"]))
            compile_classifier_model(
                model,
                learning_rate=float(trial_params["fine_tune_learning_rate"]),
                optimizer_name=trial_params["optimizer"],
                weight_decay=float(trial_params["weight_decay"]),
            )
            model.fit(
                fold_train_ds,
                validation_data=fold_val_ds,
                epochs=int(trial_params["fine_tune_epochs"]),
                callbacks=make_tuning_callbacks(
                    fold_val_ds,
                    fold_val_labels,
                    initial_lr=float(trial_params["fine_tune_learning_rate"]),
                    epochs=int(trial_params["fine_tune_epochs"]),
                    patience=2,
                ),
                class_weight=fold_class_weights,
                verbose=0,
            )

        probabilities = model.predict(fold_val_ds, verbose=0)
        macro_f1 = float(f1_score(fold_val_labels, probabilities.argmax(axis=1), average="macro"))
        fold_scores.append(macro_f1)

        # Pruning op fold-gemiddelde, niet per epoch
        mean_so_far = float(np.mean(fold_scores))
        trial.report(mean_so_far, step=fold)
        print(f"Trial {trial.number} - fold {fold + 1}/{N_FOLDS} — F1: {macro_f1:.4f} | gemiddelde: {mean_so_far:.4f}", flush=True)

        keras.backend.clear_session()
        del model, fold_train_ds, fold_val_ds
        gc.collect()

        if trial.should_prune():
            raise optuna.TrialPruned(
                f"Trial pruned after fold {fold + 1} with mean macro-F1 {mean_so_far:.4f}"
            )

    mean_f1 = float(np.mean(fold_scores))
    print(f"Trial {trial.number} — Gemiddelde F1: {mean_f1:.4f} (folds: {[round(s, 4) for s in fold_scores]})", flush=True)
    return mean_f1


FALLBACK_BEST_PARAMS = {
    "learning_rate": 5e-4,
    "dropout_rate": 0.35,
    "dense_units": 0,
    "optimizer": "adam",
    "weight_decay": 1e-4,
    "batch_size": BATCH_SIZE,
    "use_fine_tuning": True,
    "fine_tune_last_n_layers": FINAL_FINE_TUNE_LAST_N_LAYERS,
    "fine_tune_learning_rate": FINAL_FINE_TUNE_LEARNING_RATE,
    "fine_tune_epochs": FINAL_FINE_TUNE_EPOCHS,
    "head_epochs": FINAL_HEAD_EPOCHS,
}

if RUN_OPTUNA:
    sampler = optuna.samplers.TPESampler(seed=SEED)
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,   # eerste 5 trials nooit prunen
        n_warmup_steps=1,     # pruning pas vanaf fold 2
        interval_steps=1,
    )
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    optuna_best_params = study.best_params.copy()
    optuna_best_params["head_epochs"] = OPTUNA_HEAD_MAX_EPOCHS
    optuna_best_score = float(study.best_value)
    print("Best Optuna macro-F1:", optuna_best_score)
    print("Best Optuna params:", optuna_best_params)
else:
    optuna_best_params = FALLBACK_BEST_PARAMS.copy()
    optuna_best_score = None
    print("Optuna skipped; fallback params will be used.")

## 6. Final Training

In [ ]:
BEST_PARAMS = FALLBACK_BEST_PARAMS.copy()
if optuna_best_params:
    BEST_PARAMS.update(optuna_best_params)

if FINAL_FORCE_FINE_TUNING:
    BEST_PARAMS["use_fine_tuning"] = True
    BEST_PARAMS["fine_tune_last_n_layers"] = int(BEST_PARAMS.get("fine_tune_last_n_layers", FINAL_FINE_TUNE_LAST_N_LAYERS) or FINAL_FINE_TUNE_LAST_N_LAYERS)
    BEST_PARAMS["fine_tune_learning_rate"] = float(BEST_PARAMS.get("fine_tune_learning_rate", FINAL_FINE_TUNE_LEARNING_RATE) or FINAL_FINE_TUNE_LEARNING_RATE)
    BEST_PARAMS["fine_tune_epochs"] = int(BEST_PARAMS.get("fine_tune_epochs", FINAL_FINE_TUNE_EPOCHS) or FINAL_FINE_TUNE_EPOCHS)

BEST_PARAMS["head_epochs"] = FINAL_HEAD_EPOCHS
print("Final training params:")
print(pd.Series(BEST_PARAMS))

best_batch_size = int(BEST_PARAMS["batch_size"])
final_train_ds = make_dataset(train_df, batch_size=best_batch_size, shuffle=True)
final_val_ds = make_dataset(val_df, batch_size=best_batch_size)
final_class_weights = make_class_weight_dict(train_df["label"].values)
final_val_labels = val_df["label"].to_numpy()

final_model = build_model(
    num_classes=len(class_names),
    dense_units=int(BEST_PARAMS["dense_units"]),
    dropout_rate=float(BEST_PARAMS["dropout_rate"]),
    learning_rate=float(BEST_PARAMS["learning_rate"]),
    optimizer_name=BEST_PARAMS["optimizer"],
    weight_decay=float(BEST_PARAMS["weight_decay"]),
)

checkpoint_path = WORKING_ROOT / "best_lizard_model.keras"

def make_final_callbacks(initial_lr, epochs, patience=5):
    return [
        ValidationMacroF1Callback(final_val_ds, final_val_labels),
        make_cosine_decay_callback(initial_lr, epochs, verbose=1),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_macro_f1",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_macro_f1",
            mode="max",
            patience=patience,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

final_history = final_model.fit(
    final_train_ds,
    validation_data=final_val_ds,
    epochs=int(BEST_PARAMS["head_epochs"]),
    callbacks=make_final_callbacks(
        initial_lr=float(BEST_PARAMS["learning_rate"]),
        epochs=int(BEST_PARAMS["head_epochs"]),
    ),
    class_weight=final_class_weights,
    verbose=1,
)

if bool(BEST_PARAMS.get("use_fine_tuning", False)):
    set_backbone_fine_tuning(final_model, int(BEST_PARAMS["fine_tune_last_n_layers"]))
    compile_classifier_model(
        final_model,
        learning_rate=float(BEST_PARAMS["fine_tune_learning_rate"]),
        optimizer_name=BEST_PARAMS["optimizer"],
        weight_decay=float(BEST_PARAMS["weight_decay"]),
    )
    fine_tune_history = final_model.fit(
        final_train_ds,
        validation_data=final_val_ds,
        epochs=int(BEST_PARAMS["fine_tune_epochs"]),
        callbacks=make_final_callbacks(
            initial_lr=float(BEST_PARAMS["fine_tune_learning_rate"]),
            epochs=int(BEST_PARAMS["fine_tune_epochs"]),
            patience=3,
        ),
        class_weight=final_class_weights,
        verbose=1,
    )
else:
    fine_tune_history = None

if checkpoint_path.exists():
    final_model = keras.models.load_model(checkpoint_path)
    print("Loaded best checkpoint:", checkpoint_path)

In [ ]:
val_probabilities = final_model.predict(final_val_ds, verbose=0)
val_predictions = val_probabilities.argmax(axis=1)

val_accuracy = float(np.mean(val_predictions == final_val_labels))
val_macro_f1 = float(f1_score(final_val_labels, val_predictions, average="macro"))

print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation macro-F1: {val_macro_f1:.4f}")

## 7. Submission

In [ ]:
def horizontal_flip_dataset(dataset):
    return dataset.map(lambda images: tf.image.flip_left_right(images), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


test_ds = make_inference_dataset(test_df, batch_size=best_batch_size)

if USE_TTA:
    test_probabilities = (
        final_model.predict(test_ds, verbose=0) + final_model.predict(horizontal_flip_dataset(test_ds), verbose=0)
    ) / 2.0
    print("Submission inference: horizontal flip TTA")
else:
    test_probabilities = final_model.predict(test_ds, verbose=0)
    print("Submission inference: single pass")

submission = test_df[["id"]].copy()
submission["label"] = test_probabilities.argmax(axis=1).astype(int)

submission_path = WORKING_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print(submission.head())